# Milestone 4 · Notebook 1: Data Preparation

**Project:** Deepfake Voice Detection for Indian Languages · Group-3

This notebook builds the training data for Milestone 4, our final training milestone. It is the Milestone-3 data
notebook with larger numbers: the same sources at the same pinned revisions, the same speaker-disjoint split (seed 42)
and the same preprocessing (`src/preprocess.py`).

| Part | Contents | Clips |
| --- | --- | ---: |
| `train` | Kathbath (real) 4,000 + IndicSynth (fake) 4,000 per language; ASVspoof 2019 LA train: every bona fide clip + 430 per attack (A01–A06) | ~29,000 |
| `valid` | Kathbath 500 + IndicSynth 500 per language; ASVspoof dev 500 bona fide + 500 spoof | ~4,000 |
| `tune_subset` | A fixed, balanced 5,000 of `train`, for the hyperparameter sweeps | (part of `train`) |
| `train_eval` | A fixed, balanced 1,000 of `train`, scored after each epoch for the training EER | (part of `train`) |

The test sets are built separately (drop 2) and added to the same manifest.

**Output:** `data/m4/` with the processed clips and `manifests/manifest_m4.csv`, uploaded as the Kaggle dataset
`dsai-group3-m4-data`. Only a few original files per source are kept (for the before/after figures), to save space.

**Requirements:** `requirements.txt`, a Hugging Face login with access to the gated `ai4bharat/Kathbath`, ~10 GB of
disk.

## 1. Setup

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import os
import sys
import time
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfFileSystem, hf_hub_download

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO))
from src.preprocess import TARGET_SR, sha1, to_pcm16

pd.set_option("display.width", 200)
fs = HfFileSystem()

Every source is pinned to a revision, as in Milestone 3, so a re-run reads the same files.

In [2]:
SEED = 42
LANGUAGES = {"hi": "Hindi", "ta": "Tamil", "te": "Telugu"}

INDICSYNTH = "datasets/vdivyasharma/IndicSynth@c0a10386b723717aff682f757bd67f72983f269f"
KATHBATH = "datasets/ai4bharat/Kathbath@5b9e92849222026d9141acba4e8434fe816396bf"
ASVSPOOF_FULL = ("Bisher/ASVspoof_2019_LA", "aea92dd83a9c56e070c0b1e9f02e7c0d96216a4c")  # has train and dev

# Clips per language: real (Kathbath); fakes (IndicSynth) use the same numbers, half per generator
PER_LANGUAGE = {"train": 4000, "valid": 500}
ASVSPOOF_SPOOF_PER_ATTACK = {"train": 430}           # train: every bona fide clip + 430 per attack (A01-A06)
ASVSPOOF_VALID = 500                                 # dev: 500 bona fide + 500 spoof, spread over the attacks
TUNE_SUBSET, TRAIN_EVAL = 5000, 1000
CLIPS_PER_ROW_GROUP = 25    # spread the picks over many parquet row groups (and so many speakers)
RAW_PER_SOURCE = 10         # original files kept per dataset and language, for the before/after figures

DATA = REPO / "data" / "m4"
if os.environ.get("M4_SMOKE"):  # a tiny run that checks every cell, into a separate folder
    PER_LANGUAGE, ASVSPOOF_SPOOF_PER_ATTACK, ASVSPOOF_VALID = {"train": 20, "valid": 10}, {"train": 5}, 10
    TUNE_SUBSET, TRAIN_EVAL, DATA = 40, 20, REPO / "data" / "m4_smoke"
RAW, PROCESSED, MANIFESTS = DATA / "raw", DATA / "processed", DATA / "manifests"
for folder in (RAW, PROCESSED, MANIFESTS):
    folder.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
print("Data folder:", DATA)

Data folder: /home/himanshu/myproject/dsai-project/data/m4


## 2. Speaker-Disjoint Split

The same split as Milestones 2 and 3: Kathbath's training speakers (all of whom have IndicSynth fakes) are split
70 / 15 / 15 by gender with seed 42. A real clip follows its speaker; an IndicSynth fake is used only if every speaker
it involves (for FreeVC24, source and target) is in the same split.

In [3]:
meta_path = REPO / "data" / "raw" / "indicsynth" / "indicsynth_metadata_raw.parquet"
if not meta_path.exists():
    meta_path = hf_hub_download("vdivyasharma/IndicSynth", "metadata.parquet", repo_type="dataset",
                                revision=INDICSYNTH.split("@")[1])
meta = pd.read_parquet(meta_path)
meta["source_gender"] = meta.source_ref.str.split("-").str[2].str[0]

split_rng = np.random.default_rng(SEED)
speaker_rows = []
for language in LANGUAGES.values():
    part = meta[meta.language == language]
    genders = part.groupby("target_speaker").gender.first()
    for source in part.source_speaker.dropna().unique():
        if source not in genders.index:
            genders[source] = "Female" if part[part.source_speaker == source].source_gender.iloc[0] == "f" else "Male"
    for gender in ["Female", "Male"]:
        ids = np.sort(genders[genders == gender].index.to_numpy())
        split_rng.shuffle(ids)
        n_train, n_valid = round(0.70 * len(ids)), round(0.15 * len(ids))
        for i, speaker in enumerate(ids):
            split = "train" if i < n_train else ("valid" if i < n_train + n_valid else "test_indomain")
            speaker_rows.append({"language": language, "speaker_id": int(speaker), "gender": gender, "split": split})

speakers = pd.DataFrame(speaker_rows)
speakers.to_csv(MANIFESTS / "speaker_split.csv", index=False)
split_of = {(s.language, s.speaker_id): s.split for s in speakers.itertuples()}

m3 = REPO / "data" / "manifests" / "speaker_split.csv"
if m3.exists():
    print("Same split as Milestone 3:", pd.read_csv(m3).sort_values(["language", "speaker_id"]).reset_index(drop=True)
          .equals(speakers.sort_values(["language", "speaker_id"]).reset_index(drop=True)))
pd.crosstab([speakers.language, speakers.gender], speakers.split)[["train", "valid", "test_indomain"]]

Same split as Milestone 3: True


split            train  valid  test_indomain
language gender                             
Hindi    Female     37      8              8
         Male       34      7              7
Tamil    Female     22      5              5
         Male       74     16             16
Telugu   Female     29      6              6
         Male       30      6              7

## 3. Helpers

In [4]:
rows, dropped, raw_kept = [], [], Counter()


def save(clip, data, raw_name):
    """Preprocess one clip, write it as a 16-bit WAV and add a manifest row.

    clip: dict with clip_id, dataset, language, label, generator, speaker_id, ref_speaker_id, gender, text, split.
    """
    pcm, sr = to_pcm16(data)
    if pcm is None:
        dropped.append({**clip, "reason": "shorter than 1 s after trimming"})
        return
    out = PROCESSED / clip["dataset"] / clip["language"] / f"{clip['clip_id']}.wav"
    out.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out, pcm, TARGET_SR, subtype="PCM_16")
    raw_path = ""
    if raw_kept[(clip["dataset"], clip["language"])] < RAW_PER_SOURCE:  # keep a few originals
        raw_kept[(clip["dataset"], clip["language"])] += 1
        raw = RAW / clip["dataset"] / clip["language"] / raw_name
        raw.parent.mkdir(parents=True, exist_ok=True)
        raw.write_bytes(data)
        raw_path = str(raw.relative_to(DATA))
    rows.append({**clip, "path": str(out.relative_to(DATA)), "raw_path": raw_path,
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": sr, "sha1": sha1(pcm)})


def restore(clip, raw_name, orig_sr):
    """If this clip was already processed by an earlier, interrupted run, add its manifest row from the stored file.

    Returns True when the clip is already done, so it isn't downloaded again.
    """
    out = PROCESSED / clip["dataset"] / clip["language"] / f"{clip['clip_id']}.wav"
    if not out.exists():
        return False
    pcm, _ = sf.read(out, dtype="int16")
    raw = RAW / clip["dataset"] / clip["language"] / raw_name
    if raw.exists():
        raw_kept[(clip["dataset"], clip["language"])] += 1
    rows.append({**clip, "path": str(out.relative_to(DATA)), "raw_path": str(raw.relative_to(DATA)) if raw.exists() else "",
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": orig_sr, "sha1": sha1(pcm)})
    return True


def pick(candidates, n):
    """n random rows, at most CLIPS_PER_ROW_GROUP from each randomly chosen (file, row_group)."""
    groups = list(candidates.groupby(["file", "row_group"]).groups.values())
    chosen, total = [], 0
    for g in rng.permutation(len(groups)):
        take = candidates.loc[groups[g]].sample(min(CLIPS_PER_ROW_GROUP, n - total, len(groups[g])),
                                                random_state=int(rng.integers(2**31)))
        chosen.append(take)
        total += len(take)
        if total == n:
            break
    if total < n:
        print(f"  only {total} of {n} available")
    return pd.concat(chosen)


def read_columns(path, columns):
    """The given (small) columns of one parquet file, with each row's row group and position in it."""
    with fs.open(path, cache_type="none") as f:
        parquet = pq.ParquetFile(f, pre_buffer=True)
        part = parquet.read(columns=columns).to_pandas()
        sizes = [parquet.metadata.row_group(i).num_rows for i in range(parquet.metadata.num_row_groups)]
    part["file"] = path
    part["row_group"] = np.repeat(np.arange(len(sizes)), sizes)
    part["row_in_group"] = np.concatenate([np.arange(n) for n in sizes])
    return part


def read_audio(picks, column, parallel=3):
    """Yield (row, audio bytes), downloading each needed row group once.

    At most `parallel` row groups are downloaded or held in memory at a time (row groups can be large).
    """
    def load(key):
        with fs.open(key[0], cache_type="none") as f:
            table = pq.ParquetFile(f, pre_buffer=True).read_row_group(key[1], columns=[column])
        return key, table.column(column).to_pylist()

    keys = list(picks.groupby(["file", "row_group"]).groups)
    with ThreadPoolExecutor(parallel) as pool:
        pending = deque(pool.submit(load, k) for k in keys[:parallel])
        next_key = parallel
        while pending:
            (file, group), audio = pending.popleft().result()
            if next_key < len(keys):
                pending.append(pool.submit(load, keys[next_key]))
                next_key += 1
            for row in picks[(picks.file == file) & (picks.row_group == group)].itertuples():
                yield row, audio[row.row_in_group]["bytes"]
            del audio

## 4. Choosing and Preparing the Clips

### 4.1 IndicSynth (fake)

In [5]:
t0 = time.time()
meta["split"] = [split_of[(l, s)] for l, s in zip(meta.language, meta.target_speaker)]
source_split = [split_of[(l, s)] if pd.notna(s) else None for l, s in zip(meta.language, meta.source_speaker)]
usable = meta[[src is None or src == tgt for src, tgt in zip(source_split, meta.split)]].copy()
usable["file"] = INDICSYNTH + "/" + usable.language + "/" + usable.file

fake_picks = pd.concat([
    pick(usable[(usable.language == language) & (usable.generator == generator) & (usable.split == split)], n // 2)
    .assign(lang=code)
    for code, language in LANGUAGES.items() for split, n in PER_LANGUAGE.items() for generator in ["xtts_v2", "freevc24"]
])
def fake_clip(r):
    return {"clip_id": f"indicsynth_{r.lang}_{Path(r.audio_path).stem}", "dataset": "indicsynth", "language": r.lang,
            "label": "fake", "generator": r.generator, "speaker_id": r.target_speaker,
            "ref_speaker_id": r.source_speaker if pd.notna(r.source_speaker) else r.target_speaker,
            "gender": r.gender[0].lower(), "text": r.transcript if isinstance(r.transcript, str) else "",
            "split": r.split}


todo = fake_picks[[not restore(fake_clip(r), r.audio_path, 24000) for r in fake_picks.itertuples()]]
for r, data in read_audio(todo, "audio"):
    save(fake_clip(r), data, r.audio_path)
print(f"IndicSynth: {len(fake_picks):,} clips chosen ({len(todo):,} downloaded now), {(time.time() - t0) / 60:.0f} min")

IndicSynth: 13,500 clips chosen (4 downloaded now), 1 min


### 4.2 Kathbath (real)

In [6]:
t0 = time.time()
paths = [p for folder in ["hindi", "tamil", "telugu"] for p in sorted(fs.glob(f"{KATHBATH}/{folder}/train-*.parquet"))]
with ThreadPoolExecutor(4) as pool:  # listing only the small columns
    columns = ["fname", "text", "lang", "gender", "speaker_id"]
    kathbath = pd.concat(pool.map(lambda path: read_columns(path, columns), paths), ignore_index=True)
kathbath["split"] = [split_of.get((LANGUAGES[l], s)) for l, s in zip(kathbath.lang, kathbath.speaker_id)]
print(f"Kathbath train recordings: {len(kathbath):,}")

real_picks = pd.concat([pick(kathbath[(kathbath.lang == code) & (kathbath.split == split)], n)
                        for code in LANGUAGES for split, n in PER_LANGUAGE.items()])
def real_clip(r):
    return {"clip_id": f"kathbath_{r.lang}_{Path(r.fname).stem}", "dataset": "kathbath", "language": r.lang,
            "label": "real", "generator": "none", "speaker_id": r.speaker_id, "ref_speaker_id": r.speaker_id,
            "gender": r.gender[0].lower(), "text": r.text, "split": r.split}


todo = real_picks[[not restore(real_clip(r), f"{Path(r.fname).stem}.flac", 16000) for r in real_picks.itertuples()]]
for r, data in read_audio(todo, "audio_filepath"):
    save(real_clip(r), data, f"{Path(r.fname).stem}.flac")
print(f"Kathbath: {len(real_picks):,} clips chosen ({len(todo):,} downloaded now), {(time.time() - t0) / 60:.0f} min")

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ai4bharat/Kathbath/resolve/5b9e92849222026d9141acba4e8434fe816396bf/hindi/train-00009-of-00032.parquet


Retrying in 1s [Retry 1/5].


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ai4bharat/Kathbath/resolve/5b9e92849222026d9141acba4e8434fe816396bf/hindi/train-00004-of-00032.parquet


Retrying in 1s [Retry 1/5].


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ai4bharat/Kathbath/resolve/5b9e92849222026d9141acba4e8434fe816396bf/hindi/train-00007-of-00032.parquet


Retrying in 1s [Retry 1/5].


... (4 outputs left out)

Kathbath train recordings: 258,187


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ai4bharat/Kathbath/resolve/5b9e92849222026d9141acba4e8434fe816396bf/telugu/train-00022-of-00033.parquet


Retrying in 1s [Retry 1/5].


Kathbath: 13,500 clips chosen (3,809 downloaded now), 16 min


### 4.3 ASVspoof 2019 LA train and dev (English)

The official partitions are kept: train → `train`, dev → `valid`. Unlike Milestone 3, where clips were taken in blocks
and covered only some attacks, the clips are chosen from the full listing: **every** bona fide training clip plus 430
of each training attack (A01–A06), so real and fake are balanced and every attack is present.

In [7]:
t0 = time.time()
for part, split in [("train", "train"), ("validation", "valid")]:
    local = hf_hub_download(ASVSPOOF_FULL[0], f"data/{part}-00000-of-00001.parquet", repo_type="dataset",
                            revision=ASVSPOOF_FULL[1])
    parquet = pq.ParquetFile(local)
    listing = parquet.read(columns=["audio_file_name", "speaker_id", "system_id", "key"]).to_pandas()
    sizes = [parquet.metadata.row_group(i).num_rows for i in range(parquet.metadata.num_row_groups)]
    listing["row_group"] = np.repeat(np.arange(len(sizes)), sizes)
    listing["row_in_group"] = np.concatenate([np.arange(n) for n in sizes])
    bonafide, spoof = listing[listing.key == 0], listing[listing.key == 1]  # key: 0 = bona fide, 1 = spoof
    if split == "train":
        chosen = pd.concat([bonafide] + [g.sample(ASVSPOOF_SPOOF_PER_ATTACK["train"], random_state=SEED)
                                         for _, g in spoof.groupby("system_id")])
    else:
        per_attack = int(np.ceil(ASVSPOOF_VALID / spoof.system_id.nunique()))
        spoof_pick = pd.concat([g.sample(per_attack, random_state=SEED) for _, g in spoof.groupby("system_id")])
        chosen = pd.concat([bonafide.sample(ASVSPOOF_VALID, random_state=SEED),
                            spoof_pick.sample(ASVSPOOF_VALID, random_state=SEED)])
    for g, picks in chosen.groupby("row_group"):
        audio = parquet.read_row_group(int(g), columns=["audio"]).column("audio").to_pylist()
        for r in picks.itertuples():
            label = "real" if r.key == 0 else "fake"
            save({"clip_id": f"asvspoof_{r.audio_file_name}", "dataset": "asvspoof2019_la", "language": "en",
                  "label": label, "generator": r.system_id if label == "fake" else "none",
                  "speaker_id": r.speaker_id, "ref_speaker_id": r.speaker_id, "gender": "", "text": "",
                  "split": split}, audio[r.row_in_group]["bytes"], f"{r.audio_file_name}.flac")
    print(f"ASVspoof {part}: {len(chosen):,} clips chosen")
print(f"ASVspoof train and dev: {(time.time() - t0) / 60:.0f} min")

ASVspoof train: 5,160 clips chosen


ASVspoof validation: 1,000 clips chosen
ASVspoof train and dev: 1 min


## 5. The Manifest

Two fixed parts of `train` are marked, so every run uses exactly the same clips for them:

- **`tune_subset`**: 5,000 clips for the hyperparameter sweeps, with the same share of each dataset, language and label
  as `train`.
- **`train_eval`**: 1,000 clips scored after each epoch (without augmentation) for the training EER.

In [8]:
COLUMNS = ["clip_id", "path", "raw_path", "dataset", "language", "label", "generator", "speaker_id",
           "ref_speaker_id", "gender", "text", "duration_s", "orig_sr", "sha1", "split", "tune_subset", "train_eval",
           "m4_eval"]
manifest = pd.DataFrame(rows)
train = manifest[manifest.split == "train"]


def stratified(part, n, seed):
    """n clips with the same share of each (dataset, language, label) as `part`."""
    shares = part.groupby(["dataset", "language", "label"]).clip_id.count() / len(part)
    return pd.concat([part[(part.dataset == d) & (part.language == l) & (part.label == y)]
                      .sample(int(round(share * n)), random_state=seed)
                      for (d, l, y), share in shares.items()]).clip_id


manifest["tune_subset"] = manifest.clip_id.isin(stratified(train, TUNE_SUBSET, SEED))
manifest["train_eval"] = manifest.clip_id.isin(stratified(train, TRAIN_EVAL, SEED + 1))
manifest["m4_eval"] = False   # set for the test splits in drop 2
manifest = manifest[COLUMNS].sort_values(["split", "dataset", "language", "clip_id"])
manifest.to_csv(MANIFESTS / "manifest_m4.csv", index=False)
pd.DataFrame(dropped).to_csv(MANIFESTS / "dropped.csv", index=False)
print(f"{len(manifest):,} clips kept, {len(dropped)} dropped, {manifest.duration_s.sum() / 3600:.1f} h after preprocessing")

32,981 clips kept, 179 dropped, 52.6 h after preprocessing


### Clips per split, dataset and label

In [9]:
manifest.pivot_table(index=["split", "dataset"], columns="label", values="clip_id", aggfunc="count", margins=True,
                     margins_name="total")

label                     fake     real  total
split dataset                                 
train asvspoof2019_la   2498.0   2525.0   5023
      indicsynth       11996.0      NaN  11996
      kathbath             NaN  12000.0  12000
valid asvspoof2019_la    484.0    478.0    962
      indicsynth        1500.0      NaN   1500
      kathbath             NaN   1500.0   1500
total                  16478.0  16503.0  32981

### Clips per split, language and generator

In [10]:
manifest.pivot_table(index=["split", "language"], columns="generator", values="clip_id", aggfunc="count", fill_value=0)

generator       A01  A02  A03  A04  A05  A06  freevc24  none  xtts_v2
split language                                                       
train en        416  412  411  412  419  428         0  2525        0
      hi          0    0    0    0    0    0      2000  4000     1999
      ta          0    0    0    0    0    0      1998  4000     2000
      te          0    0    0    0    0    0      2000  4000     1999
valid en         82   80   80   81   78   83         0   478        0
      hi          0    0    0    0    0    0       250   500      250
      ta          0    0    0    0    0    0       250   500      250
      te          0    0    0    0    0    0       250   500      250

### The tuning subset and the training-EER clips

In [11]:
pd.DataFrame({name: manifest[manifest[name]].groupby(["dataset", "label"]).clip_id.count()
              for name in ["tune_subset", "train_eval"]})

tune_subset  train_eval
dataset         label                         
asvspoof2019_la fake           430          86
                real           435          87
indicsynth      fake          2067         414
kathbath        real          2067         414

### Dropped clips and original sample rates

In [12]:
print(pd.DataFrame(dropped).groupby(["dataset", "split"]).size().rename("dropped") if dropped else "No clips dropped")
manifest.groupby("dataset").orig_sr.value_counts().unstack(fill_value=0).rename(columns=lambda sr: f"{sr / 1000:g} kHz")

dataset          split
asvspoof2019_la  train    137
                 valid     38
indicsynth       train      4
Name: dropped, dtype: int64


orig_sr,16 kHz,24 kHz
dataset,,
asvspoof2019_la,5985,0
indicsynth,0,13496
kathbath,13500,0


## 6. Leakage Checks

No speaker may be in two splits (including a FreeVC24 clip's source speaker), no clip may appear twice, and the
ASVspoof partitions must not share speakers.

In [13]:
def speaker_set(split):
    part = manifest[(manifest.split == split) & manifest.language.isin(LANGUAGES)]
    return set(part.speaker_id.astype(str)) | set(part.ref_speaker_id.astype(str))


asv = manifest[manifest.dataset == "asvspoof2019_la"]
asv_speakers = [set(asv[asv.split == s].speaker_id) for s in asv.split.unique()]
leakage = pd.DataFrame([
    ("speakers in both train and valid", len(speaker_set("train") & speaker_set("valid"))),
    ("duplicate clips (same SHA-1)", int(manifest.sha1.duplicated().sum())),
    ("ASVspoof speakers in two partitions",
     sum(len(a & b) for i, a in enumerate(asv_speakers) for b in asv_speakers[i + 1:])),
], columns=["check", "count"])
leakage.to_csv(MANIFESTS / "leakage_report.csv", index=False)
leakage

,check,count
0,speakers in both train and valid,0
1,duplicate clips (same SHA-1),0
2,ASVspoof speakers in two partitions,0


## 7. Folder Structure

In [14]:
for folder in sorted(p for p in DATA.glob("*/*") if p.is_dir()):
    files = [f for f in folder.rglob("*") if f.is_file()]
    print(f"{str(folder.relative_to(DATA)):35s} {len(files):6,} files {sum(f.stat().st_size for f in files) / 1e9:6.2f} GB")

processed/asvspoof2019_la            5,985 files   0.48 GB


processed/indicsynth                13,496 files   2.94 GB


processed/kathbath                  13,500 files   2.64 GB
raw/asvspoof2019_la                     10 files   0.00 GB
raw/indicsynth                          30 files   0.01 GB
raw/kathbath                            30 files   0.01 GB


## 8. Summary

`data/m4/` now holds the training data for Milestone 4 (`train`, `valid`, the tuning subset and the training-EER
clips) with its manifest and leakage report. It is uploaded as the Kaggle dataset `dsai-group3-m4-data` and shared with
the team; `02_training.ipynb` reads it. The test sets are added in drop 2.